# 5 · DE CUDA, version 1 : un thread par individu

La version 1 est la traduction directe du DE séquentiel sur le GPU : même algorithme (DE/rand/1/bin,
F = 0,5, CR = 0,3), mêmes fonctions, même budget de 10⁴·D évaluations et même format de résultats. Chaque
thread s'occupe d'un individu entier. Le code est dans `src/cuda/de_cuda_v1.cu` et `src/cuda/cuda_utils.h`.

| Kernel | Grille | Rôle |
|---|---|---|
| `k_init_rng` | (N+T−1)/T blocs, 1 thread par individu | un générateur cuRAND (Philox) par individu, graine du run |
| `k_init_pop` | idem | population uniforme dans le domaine |
| `k_eval` | idem | `fit[i] = evaluate(f, x_i, o, D)` — la fonction `__host__ __device__` de l'interface |
| `k_mutation_croisement` | idem | `trial[i]` = croisement de x_i et de x_r1 + F (x_r2 − x_r3), bornes respectées |
| `k_selection` | idem | si f(trial_i) ≤ f(x_i), x_i ← trial_i |

- **Mémoire** : `pop`, `trial`, `fit`, `fit_trial` et les générateurs en mémoire globale ; le décalage `o` en **mémoire constante** (tous les threads lisent `o[j]` au même moment : une seule lecture diffusée à tout le warp).
- **Aucun transfert dans la boucle** : une seule copie GPU → CPU à la fin (les N fitness).
- **Pas de race condition** : les essais sont écrits dans `trial`, la sélection n'a lieu qu'après, dans un autre kernel.
- **Erreurs** : `CUDA_CHECK` sur chaque appel CUDA, `CUDA_CHECK_KERNEL()` (`cudaGetLastError`) après chaque lancement.
- **Chronométrage** : `cudaFree(0)` crée le contexte avant le chrono, puis `cudaEvent` autour de tout l'algorithme (initialisation comprise, comme le séquentiel).

Dans ce notebook, nous compilons cette version sans aucun warning, nous vérifions qu'une erreur CUDA
provoquée volontairement est bien détectée, que la même graine donne le même résultat et que le temps
varie très peu d'un run à l'autre. Nous la lançons enfin sur les 4 fonctions pour D = 10, 50 et 100 :
chaque run respecte exactement le budget et passe ses contrôles.

In [1]:
# Récupérer le projet.
# Si DE_CUDA.zip a été déposé dans Colab (icône dossier à gauche → Importer), on utilise ce dossier ;
# sinon on copie le dépôt public GitHub (aucun compte nécessaire).
import os
if os.path.exists("/content/DE_CUDA.zip") and not os.path.exists("/content/DE_CUDA/src"):
    !unzip -q -o /content/DE_CUDA.zip -d /content
if not os.path.exists("/content/DE_CUDA/src"):
    !git clone -q https://github.com/amenacer/DE_CUDA.git /content/DE_CUDA
%cd /content/DE_CUDA
!nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv

/content/DE_CUDA
name, driver_version, memory.total [MiB]
Tesla T4, 580.82.07, 15360 MiB


In [2]:
import subprocess, re, statistics as st, os, json
ROOT = "/content/DE_CUDA"
RAPPORT = []                       # tout ce qui est affiche est aussi garde pour le rapport
def note(s=""):
    print(s); RAPPORT.append(str(s))

def run(args, cwd=ROOT, ok_codes=(0,)):
    p = subprocess.run(args, cwd=cwd, capture_output=True, text=True)
    if p.returncode not in ok_codes:
        print(p.stdout); print(p.stderr)
        raise RuntimeError(f"{args[0]} a renvoye {p.returncode}")
    return p

def compile_cuda(src, out):
    p = run(["nvcc", "-O3", "-arch=sm_75", "-Xcompiler", "-Wall", "-Xcompiler", "-Wextra",
             "-o", out, src], cwd=f"{ROOT}/src/cuda")
    warn = (p.stdout + p.stderr).strip()
    note(f"{src} : " + ("compilation SANS aucun warning" if not warn else "WARNINGS :\n" + warn))

KV = re.compile(r"(\w+)=(\S+)")
def lance(prog, f, D, N, seed, csv=None, extra=()):
    """Lance un run et renvoie le dictionnaire de sa derniere ligne (best_error, time_s, ...)."""
    args = [prog, f, str(D), str(N), str(seed)] + ([csv] if csv else []) + list(extra)
    out = run(args).stdout.strip().splitlines()[-1]
    d = dict(KV.findall(out))
    for k in ("best_error", "time_s"):
        d[k] = float(d[k])
    return d

def mann_whitney(a, b):
    """p-valeur du test de Mann-Whitney bilateral ; 1.0 si les deux series sont identiques et constantes."""
    from scipy.stats import mannwhitneyu
    if len(set(a) | set(b)) == 1:
        return 1.0
    return float(mannwhitneyu(a, b, alternative="two-sided").pvalue)

def tableau_md(lignes, entetes):
    s = "| " + " | ".join(entetes) + " |\n|" + "---|" * len(entetes) + "\n"
    for l in lignes:
        s += "| " + " | ".join(str(x) for x in l) + " |\n"
    return s
print("outils prets")

outils prets


## 1. Compiler sans warning

Compilation avec `-Wall -Wextra` : aucun warning attendu.

In [3]:
compile_cuda("de_cuda_v1.cu", "de_cuda_v1")
V1 = f"{ROOT}/src/cuda/de_cuda_v1"

de_cuda_v1.cu : compilation SANS aucun warning


## 2. Détection d'une erreur CUDA

`--test-erreur` lance volontairement un kernel avec 2048 threads par bloc (le maximum est 1024). `CUDA_CHECK_KERNEL` doit arrêter le programme avec un message clair : fichier, ligne et erreur renvoyée par CUDA.

In [4]:
p = run([V1, "--test-erreur"], ok_codes=(0, 1))
note(p.stdout.strip()); note(p.stderr.strip())
note(f"code de retour : {p.returncode}  =>  " + ("erreur detectee : OK" if p.returncode == 1 and "Erreur CUDA" in p.stderr else "ECHEC"))

Lancement de k_eval avec 2048 threads par bloc (maximum autorise : 1024)...
Erreur CUDA de_cuda_v1.cu:137 : invalid argument
  dans : cudaGetLastError()
code de retour : 1  =>  erreur detectee : OK


## 3. Même graine → même résultat

Deux runs avec la même graine doivent donner exactement la même erreur ; une autre graine donne un autre résultat.

In [5]:
note("Determinisme (D = 10, N = 50)")
for f in ["sphere", "rastrigin", "rosenbrock", "griewank"]:
    a = lance(V1, f, 10, 50, 7); b = lance(V1, f, 10, 50, 7); c = lance(V1, f, 10, 50, 8)
    same = a["best_error"] == b["best_error"]
    note(f"  {f:10s} seed 7 : {a['best_error']:.15e} | seed 7 : {b['best_error']:.15e} "
         f"-> {'IDENTIQUES' if same else 'DIFFERENTS'} | seed 8 : {c['best_error']:.6e}")

Determinisme (D = 10, N = 50)
  sphere     seed 7 : 0.000000000000000e+00 | seed 7 : 0.000000000000000e+00 -> IDENTIQUES | seed 8 : 0.000000e+00
  rastrigin  seed 7 : 0.000000000000000e+00 | seed 7 : 0.000000000000000e+00 -> IDENTIQUES | seed 8 : 0.000000e+00
  rosenbrock seed 7 : 3.506597420000000e-02 | seed 7 : 3.506597420000000e-02 -> IDENTIQUES | seed 8 : 1.157099e+00
  griewank   seed 7 : 0.000000000000000e+00 | seed 7 : 0.000000000000000e+00 -> IDENTIQUES | seed 8 : 0.000000e+00


## 4. Stabilité du chronométrage

Rastrigin, D = 100, N = 100, même graine, 5 runs : l'écart de temps entre deux runs identiques doit rester sous 10 %.

In [6]:
t = [lance(V1, "rastrigin", 100, 100, 1)["time_s"] for _ in range(5)]
e2 = abs(t[1] - t[0]) / min(t[0], t[1]) * 100
emax = (max(t) - min(t)) / min(t) * 100
note("Temps (s) de 5 runs identiques : " + ", ".join(f"{x:.4f}" for x in t))
note(f"ecart run1/run2 = {e2:.2f} %   ecart max/min sur 5 runs = {emax:.2f} %  =>  " + ("OK (< 10 %)" if e2 < 10 else "ECHEC"))

Temps (s) de 5 runs identiques : 1.8403, 1.8381, 1.8377, 1.8358, 1.8348
ecart run1/run2 = 0.12 %   ecart max/min sur 5 runs = 0.30 %  =>  OK (< 10 %)


## 5. Les 4 fonctions, D = 10, 50, 100 (N = 100)

Chaque run doit finir avec `sante=OK` et exactement 10⁴·D évaluations.

In [7]:
lignes = []
for f in ["sphere", "rosenbrock", "griewank", "rastrigin"]:
    for D in (10, 50, 100):
        r = lance(V1, f, D, 100, 1)
        lignes.append([f, D, 100, f"{r['best_error']:.3e}", r["FE"], f"{r['time_s']:.4f}", r["sante"]])
note(tableau_md(lignes, ["fonction", "D", "N", "best_error", "FE", "temps (s)", "sante"]))

| fonction | D | N | best_error | FE | temps (s) | sante |
|---|---|---|---|---|---|---|
| sphere | 10 | 100 | 0.000e+00 | 100000 | 0.0156 | OK |
| sphere | 50 | 100 | 0.000e+00 | 500000 | 0.2523 | OK |
| sphere | 100 | 100 | 0.000e+00 | 1000000 | 0.8643 | OK |
| rosenbrock | 10 | 100 | 5.849e+00 | 100000 | 0.0181 | OK |
| rosenbrock | 50 | 100 | 4.118e+01 | 500000 | 0.3163 | OK |
| rosenbrock | 100 | 100 | 8.961e+01 | 1000000 | 1.1574 | OK |
| griewank | 10 | 100 | 0.000e+00 | 100000 | 0.0360 | OK |
| griewank | 50 | 100 | 0.000e+00 | 500000 | 0.7525 | OK |
| griewank | 100 | 100 | 0.000e+00 | 1000000 | 2.8823 | OK |
| rastrigin | 10 | 100 | 0.000e+00 | 100000 | 0.0262 | OK |
| rastrigin | 50 | 100 | 1.729e+02 | 500000 | 0.4808 | OK |
| rastrigin | 100 | 100 | 5.336e+02 | 1000000 | 1.8350 | OK |



## 6. Rapport

La dernière cellule enregistre les résultats ci-dessus dans `results/etape3_v1_controles.md`.

In [8]:
os.makedirs(f"{ROOT}/results", exist_ok=True)
with open(f"{ROOT}/results/etape3_v1_controles.md", "w") as fp:
    fp.write("# Étape 3 — contrôles du DE CUDA v1\n\n")
    fp.write("GPU : " + run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"]).stdout.strip() + "\n\n")
    fp.write("```\n" + "\n".join(RAPPORT) + "\n```\n")